In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SimpleCNN(nn.Module):
    def __init__(self,num_classes=4):
        super(SimpleCNN,self).__init__() #(1,64,64)
        self.conv1= nn.Conv2d(in_channels=1,out_channels=16,stride=2,kernel_size=3,padding=1)
        self.conv2= nn.Conv2d(in_channels=16,out_channels=32,stride=2,kernel_size=3,padding=1)
        self.pool= nn.MaxPool2d(2,2)
        self.fc1=nn.Linear(32*8*8,128)
        self.fc2=nn.Linear(128,num_classes)
        # self.sf = nn.Softmax(dim = 1)
        self.relu= nn.ReLU()
        
    def forward(self,x):
        x=self.relu(self.conv1(x))
        # print(x.shape)
        x=self.pool(self.conv2(x))
        # print(x.shape)
        x=x.view(x.size(0),-1)
        # print(x.shape)
        x=self.relu(self.fc1(x))
        # print(x.shape)
        x=self.fc2(x)
        return x

model= SimpleCNN()
print(model)


SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
  (relu): ReLU()
)


In [19]:
train_data = [torch.randn(100,1,64,64),torch.randint(1,4,(100,))]
print(train_data[1].shape)
test_data =[torch.randn(100,1,64,64),torch.randint(1,4,(100,))]
print(test_data[1].shape)

from torch.utils.data import TensorDataset,DataLoader
train_dataset =TensorDataset(train_data[0],train_data[1])
test_dataset = TensorDataset(test_data[0],test_data[1])

train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)


torch.Size([100])
torch.Size([100])


In [21]:
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(model.parameters(),lr=0.001)

In [33]:
epochs =5
for epoch in range(epochs):
    for batch_idx,(images,labels) in enumerate(train_loader):
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # print(f"Epoch Loss: {loss:.4f}")
    print(f"Epoch Loss: {loss:.4f}")




Epoch Loss: 0.4504
Epoch Loss: 0.3679
Epoch Loss: 0.4236
Epoch Loss: 0.1986
Epoch Loss: 0.1297


In [35]:
#test over test_loader
model.eval()
correct=0
totals=0
with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        outputs = model(images)

        _,predicted = torch.max(outputs,1)
        totals +=labels.size(0)
        correct+=(predicted==labels).sum().item()
        print(f"\n[DEBUG] Batch {batch_idx+1}")
        print("Images shape:", images.shape)
        print("Outputs shape:", outputs.shape)
        print("Predicted:", predicted[:5])
        print("Actual   :", labels[:5])
accuracy=100*correct/totals
print(f"\nTest Accuracy: {accuracy:.2f}")
print(*predicted)
      


[DEBUG] Batch 1
Images shape: torch.Size([32, 1, 64, 64])
Outputs shape: torch.Size([32, 4])
Predicted: tensor([2, 2, 1, 1, 1])
Actual   : tensor([1, 1, 2, 1, 2])

[DEBUG] Batch 2
Images shape: torch.Size([32, 1, 64, 64])
Outputs shape: torch.Size([32, 4])
Predicted: tensor([1, 1, 1, 2, 1])
Actual   : tensor([2, 1, 3, 2, 1])

[DEBUG] Batch 3
Images shape: torch.Size([32, 1, 64, 64])
Outputs shape: torch.Size([32, 4])
Predicted: tensor([1, 1, 1, 1, 3])
Actual   : tensor([1, 2, 1, 1, 3])

[DEBUG] Batch 4
Images shape: torch.Size([4, 1, 64, 64])
Outputs shape: torch.Size([4, 4])
Predicted: tensor([1, 1, 1, 1])
Actual   : tensor([1, 2, 3, 2])

Test Accuracy: 36.00
tensor(1) tensor(1) tensor(1) tensor(1)
